# Install Packages

In [ ]:
# Install torch metrics package for LOSS, EVALUATION METRICS
!pip install lightning-utilities
!pip install torchmetrics --no-deps
!pip install torchinfo
!pip install -U albumentations

In [ ]:
import os
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix


import torch
import torch.nn as nn
import torchvision

from torchvision.transforms import v2
from torchvision.transforms.functional import to_pil_image

from torchmetrics import MeanMetric, Accuracy
from torchmetrics import ConfusionMatrix, Precision, Recall, F1Score
from torchinfo import summary

import torch
from torch.utils.data import random_split, DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import albumentations as alb
from albumentations.pytorch import ToTensorV2
import gc
import copy

In [ ]:
print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU detected")


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print("Device:", device)

#Albumentations

resizing to 384

In [ ]:
import albumentations as alb
from albumentations.pytorch import ToTensorV2

train_augmentations = {
    "DF": alb.Compose([
        alb.HorizontalFlip(p=0.7),
        alb.Rotate(limit=270, p=0.7),
        alb.RandomBrightnessContrast(p=0.7),
        alb.CoarseDropout(max_holes=2, max_height=18, max_width=18, p=0.3),
        alb.GaussianBlur(p=0.2),
        alb.Resize(384,384),
        alb.Normalize(mean=[0.485, 0.456, 0.406],
                  std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ]),
    "VASC": alb.Compose([
        alb.HorizontalFlip(p=0.7),
        alb.Rotate(limit=270, p=0.7),
        alb.RandomBrightnessContrast(p=0.7),
        alb.CoarseDropout(max_holes=2, max_height=18, max_width=18, p=0.3),
        alb.GaussianBlur(p=0.2),
        alb.Resize(384,384),
        alb.Normalize(mean=[0.485, 0.456, 0.406],
                  std=[0.229, 0.224, 0.225]),
        ToTensorV2()

    ]),
    "AK": alb.Compose([
        alb.HorizontalFlip(p=0.7),
        alb.Rotate(limit=270, p=0.7),
        alb.RandomBrightnessContrast(p=0.7),
        alb.CoarseDropout(max_holes=2, max_height=18, max_width=18, p=0.3),
        alb.Resize(384,384),
        alb.Normalize(mean=[0.485, 0.456, 0.406],
                  std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ]),
    "SCC": alb.Compose([
        alb.HorizontalFlip(p=0.7),
        alb.Rotate(limit=270, p=0.7),
        alb.RandomBrightnessContrast(p=0.3),
        alb.CoarseDropout(max_holes=2, max_height=18, max_width=18, p=0.3),
        alb.GaussianBlur(p=0.2),
        alb.Resize(384,384),
        alb.Normalize(mean=[0.485, 0.456, 0.406],
                  std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ]),
    "BKL":alb.Compose([
        alb.HorizontalFlip(p=0.5),
        alb.Rotate(limit=270, p=0.5),
        alb.RandomBrightnessContrast(p=0.5),
        alb.Resize(384,384),
        alb.Normalize(mean=[0.485, 0.456, 0.406],
                  std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ]),
    "BCC":alb.Compose([
        alb.HorizontalFlip(p=0.3),
        alb.Rotate(limit=270, p=0.5),
        alb.RandomBrightnessContrast(p=0.5),
        alb.Resize(384,384),
        alb.Normalize(mean=[0.485, 0.456, 0.406],
                  std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ]),
    "MEL":alb.Compose([
        alb.HorizontalFlip(p=0.3),
        alb.RandomBrightnessContrast(p=0.4),
        alb.Resize(384,384),
        alb.Normalize(mean=[0.485, 0.456, 0.406],
                  std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ]),
    "NV":alb.Compose([
        alb.Resize(384,384),
        alb.Normalize(mean=[0.485, 0.456, 0.406],
                  std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
}

In [ ]:
class AlbTransform:
    def __init__(self, augmentation_dict):
        self.augmentation_dict = augmentation_dict

    def __call__(self, img):
        img = np.array(img)  # Convert PIL Image to NumPy array
        label = img.filename.split("/")[-2]  # Extract label from folder structure

        if label in self.augmentation_dict:
            augmented = self.augmentation_dict[label](image=img)["image"]
        else:
            augmented = img  # No augmentation applied

        tensor_img = ToTensorV2()(image=augmented)["image"]
        tensor_img = tensor_img.permute(2, 0, 1).contiguous()  # Fix channel order

        return tensor_img


# Train/Val Set

In [ ]:
from torch.utils.data import Subset
from collections import Counter
import torch

# Load entire dataset once
dataset = ImageFolder(root="/content/drive/MyDrive/ISIC_2019_raw", transform=None)

# Shuffle indices manually for randomness
indices = torch.randperm(len(dataset)).tolist()

# Define split sizes
train_size = int(0.85 * len(dataset))
val_size = len(dataset) - train_size

# Ensure validation set is different
train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_set = Subset(dataset, train_indices)
val_set = Subset(dataset, val_indices)

In [ ]:
train_labels = [dataset.targets[i] for i in train_indices]
original_counts = Counter(train_labels)

In [ ]:
 # Define classes
classes = ["AK", "BCC", "BKL","DF","MEL","NV","SCC","VASC"]
num_class=len(classes)
print("Number of Classes:", num_class)

In [ ]:
original_classes = ["AK", "BCC", "BKL","DF","MEL","NV","SCC","VASC"]
original_vals = [original_counts.get(i, 0) for i in range(len(classes))]

In [ ]:
# Calc inverse freq sample weights
class_counts = Counter(train_labels)
class_weights_dict = {cls: 1.0/count for cls, count in class_counts.items()}
sample_weights = torch.tensor([class_weights_dict[label] for label in train_labels], dtype=torch.float)


In [ ]:
# Sampler
from torch.utils.data import WeightedRandomSampler
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights),replacement=True)

In [ ]:
train_set.dataset.transform = AlbTransform(train_augmentations)  # Augmentations only for training
val_set.dataset.transform = transforms.Compose([
    transforms.Resize((384,384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])  #  No augmentations for validation


In [ ]:
test_set=ImageFolder(root="/content/drive/MyDrive/ISIC_2019_test_raw", transform=transforms.Compose([
    transforms.Resize((384,384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]))

reduced from 64 as increased computational power for resizing to 384x384

In [ ]:
train_loader = DataLoader(train_set, batch_size=8, sampler=sampler)
val_loader = DataLoader(val_set, batch_size=8, shuffle=False)
test_loader=DataLoader(test_set, batch_size=8, shuffle=False)

In [ ]:
print("Number of Train Samples:", len(train_set))
print("Number of Validation Samples:", len(val_set))
print("Number of Test Samples:", len(test_set))


In [ ]:
 # Define classes
classes = ["AK", "BCC", "BKL","DF","MEL","NV","SCC","VASC"]
num_class=len(classes)
print("Number of Classes:", num_class)

# Augmented Count

In [ ]:
augmented_counts = Counter()
for images, labels in train_loader:
  for label in labels:
    augmented_counts[int(label)] += 1

augmented_val = [augmented_counts.get(i,0) for i in range(len(classes))]

# Plot

In [ ]:
sns.set(style='whitegrid')
x=range(len(classes))
bar_width=0.35

plt.figure(figsize=(12,6))
plt.bar(x, original_vals, width=bar_width, label='Original', color='skyblue')
plt.bar([i + bar_width for i in x], augmented_val, width=bar_width, label='Augmented', color='salmon')

plt.xlabel('Class')
plt.ylabel('Number of Samples')
plt.title('Class Distribution Before and After Augmentation For Training Set')
plt.xticks([i + bar_width / 2 for i in x], classes, rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Thesis/class_dist_compare.png')
plt.show()

In [ ]:
# Map class index to name
class_map = {i: name for i, name in enumerate(classes)}

# Extract labels
train_labels = [dataset.targets[i] for i in train_indices]
val_labels = [dataset.targets[i] for i in val_indices]
test_labels = [sample[1] for sample in test_set.samples]  # test_set is a full ImageFolder

# Count frequencies
train_counts = Counter(train_labels)
val_counts = Counter(val_labels)
test_counts = Counter(test_labels)

# Format for plotting
x = list(range(num_class))
bar_width = 0.25

train_vals = [train_counts.get(i, 0) for i in x]
val_vals = [val_counts.get(i, 0) for i in x]
test_vals = [test_counts.get(i, 0) for i in x]

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar([i - bar_width for i in x], train_vals, width=bar_width, label="Train", color="skyblue")
plt.bar(x, val_vals, width=bar_width, label="Validation", color="lightgreen")
plt.bar([i + bar_width for i in x], test_vals, width=bar_width, label="Test", color="salmon")

plt.xticks(ticks=x, labels=classes, rotation=45)
plt.ylabel("Number of Samples")
plt.title("Sample Distribution per Class in Train / Validation / Test Sets")
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Training set
plt.figure(figsize=(8, 4))
plt.bar(classes, train_vals, color='steelblue')
plt.title("Training Set Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Validation set
plt.figure(figsize=(8, 4))
plt.bar(classes, val_vals, color='seagreen')
plt.title("Validation Set Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Test set
plt.figure(figsize=(8, 4))
plt.bar(classes, test_vals, color='indianred')
plt.title("Test Set Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


#Raw Images Visulaisation

In [ ]:
def one_image_per_class(dataset, classes):
  shown_classes = set()
  fig=plt.figure(figsize=(12,12))
  idx=1

  for path, class_idx in dataset.samples:
    class_name=classes[class_idx]
    if class_name not in shown_classes:
      img = Image.open(path).convert("RGB")
      ax=fig.add_subplot(4,4,idx)
      ax.imshow(img)
      ax.set_title(class_name)
      ax.axis('off')
      shown_classes.add(class_name)
      idx+=1
    if len(shown_classes) == len(classes):
      break

  plt.tight_layout()
  plt.show()

one_image_per_class(dataset, classes)

#Augmented Images

In [ ]:
augmentation_map = {
        "HorizontalFlip": "Horizontal Flip",
        "Rotate": "Rotation",
        "RandomBrightnessContrast": "Brightness & Contrast",
        "CoarseDropout": "Coarse Dropout",
        "GaussianBlur": "Gaussian Blur"
}

In [ ]:
def readable_augmentations(pipeline):
  readable = []
  for t in pipeline.transforms:
    name=type(t).__name__
    if name in augmentation_map:
      readable.append(augmentation_map[name])
  return readable

In [ ]:
import cv2
def visualize_augmentations(image_path, class_label, augmentations, n=4):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, n + 1, figsize=(15, 5))
    axes[0].imshow(image)
    axes[0].set_title(f"Original {class_label}")
    axes[0].axis('off')

    aug_pipeline = augmentations[class_label]

    for i in range(n):
        augmented = aug_pipeline(image=image)['image']
        # Convert tensor to numpy if needed
        if isinstance(augmented, torch.Tensor):
            augmented = augmented.permute(1, 2, 0).numpy()
            augmented = (augmented * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]  # Unnormalize
            augmented = (augmented * 255).astype('uint8')
        transform_names = [type(t).__name__ for t in aug_pipeline.transforms if not isinstance(t, (ToTensorV2, alb.Normalize, alb.Resize))]
        bullet_list = "\n".join(f"• {name}" for name in transform_names)
        title_text = f"Augmented {i+1}\n{bullet_list}"


        axes[i + 1].imshow(augmented)
        axes[i + 1].set_title(title_text, fontsize=10)
        axes[i + 1].axis('off')

    plt.tight_layout()
    plt.show()


### Visualise aug of each class

In [ ]:
visualize_augmentations("/content/drive/MyDrive/ISIC_2019_raw/AK/ISIC_0024654.jpg", "AK", train_augmentations, n=3)


In [ ]:
visualize_augmentations("/content/drive/MyDrive/ISIC_2019_raw/BCC/ISIC_0024411.jpg", "BCC", train_augmentations, n=3)


In [ ]:
visualize_augmentations("//content/drive/MyDrive/ISIC_2019_raw/BKL/ISIC_0012136_downsampled.jpg", "BKL", train_augmentations, n=3)


In [ ]:
visualize_augmentations("/content/drive/MyDrive/ISIC_2019_raw/DF/ISIC_0024396.jpg", "DF", train_augmentations, n=3)


In [ ]:
visualize_augmentations("/content/drive/MyDrive/ISIC_2019_raw/MEL/ISIC_0000029_downsampled.jpg", "MEL", train_augmentations, n=3)


In [ ]:
visualize_augmentations("/content/drive/MyDrive/ISIC_2019_raw/NV/ISIC_0000008.jpg", "NV", train_augmentations, n=3)


In [ ]:
visualize_augmentations("/content/drive/MyDrive/ISIC_2019_raw/SCC/ISIC_0024522.jpg", "SCC", train_augmentations, n=3)


In [ ]:
visualize_augmentations("/content/drive/MyDrive/ISIC_2019_raw/VASC/ISIC_0024747.jpg", "VASC", train_augmentations, n=3)


In [ ]:
def visualize_individual_augmentations(image_path, class_label, augmentations, augmentation_map):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Get the pipeline for the class
    aug_pipeline = augmentations[class_label]

    # Filter out non-visual transforms
    visual_transforms = [t for t in aug_pipeline.transforms if not isinstance(t, (ToTensorV2, alb.Normalize, alb.Resize))]

    fig, axes = plt.subplots(1, len(visual_transforms) + 1, figsize=(5 * (len(visual_transforms) + 1), 5))
    axes[0].imshow(image)
    axes[0].set_title(f"Original {class_label}", fontsize=12, pad=10)
    axes[0].axis('off')

    for i, transform in enumerate(visual_transforms):
        # Apply single transform
        single_aug = alb.Compose([
            transform,
            alb.Resize(224, 224),  # Ensure consistent size
        ])
        augmented = single_aug(image=image)['image']

        # Convert to uint8 if needed
        if isinstance(augmented, torch.Tensor):
            augmented = augmented.permute(1, 2, 0).numpy()
            augmented = (augmented * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
            augmented = (augmented * 255).astype('uint8')

        # Get readable name
        transform_name = type(transform).__name__
        readable_name = augmentation_map.get(transform_name, transform_name)

        axes[i + 1].imshow(augmented)
        axes[i + 1].set_title(readable_name, fontsize=11, pad=10)
        axes[i + 1].axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
visualize_augmentations("/content/drive/MyDrive/ISIC_2019_raw/AK/ISIC_0024654.jpg", "AK", train_augmentations, augmentation_map)

#Pretrained Models

Next steps
1. Load Pretrained Models
2. Maybe make my own?/Contrastive Learning Model and compare
    - MoCo (classifier fine tune)
    - would need contrastive learning loss function for pretraining
    - Compare contrastive embeddings vs CNN

3. Evaluate Metrics F1, acc, prec, auc
4. Grad-CAM/SHAP interpretability inspected

5. Fusion Model using patient metadata
6. Evaluation metrics F1, acc,prec, auc
7. Grad-CAM/SHAP intrepretability inspected

8. Compare Overall results
  - Img ONLY vs Img + Metadata:
    - F1 scores, reliability (precision)
    - Interpretability (heatmap)
    - Interpretability (SHAP)



In [ ]:
import torchvision.models as models

In [ ]:
project_root = "/content/drive/MyDrive/Thesis/DaBIGone"
os.makedirs(project_root, exist_ok=True)

In [ ]:
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score, MulticlassAUROC

## Resnet

In [ ]:
model_name = "ResNet50"
checkpoint_path = os.path.join(project_root, f"{model_name}_checkpoint_latest.pth")
train_val_path = os.path.join(project_root, f"{model_name}_train_val_results.csv")
test_results_path = os.path.join(project_root, f"{model_name}_test_results.csv")

In [ ]:
res_model18=models.resnet18(pretrained=True)

In [ ]:
res_model18.fc= nn.Sequential(
    nn.Dropout(p=0.6), # Look at changing to 0.6
    nn.Linear(res_model18.fc.in_features, num_class))

In [ ]:
res_model50 = models.resnet50(pretrained=True)

In [ ]:
res_model50.fc= nn.Sequential(
    nn.Dropout(p=0.6), # Look at changing to 0.6
    nn.Linear(res_model50.fc.in_features, num_class))

## Vision Transformers

used for medical imaging
- robust to dataset imbalance

In [ ]:
import timm
vis_model = timm.create_model("vit_base_patch16_224", pretrained=True)

In [ ]:
print(vis_model)

In [ ]:
vis_model.head = nn.Linear(vis_model.head.in_features, num_class)

## Efficient Net

excellent accuracy balance depth width and resolution

In [ ]:
import timm
eff_model = timm.create_model("efficientnet_b3",pretrained=True)

In [ ]:
eff_model.classifier =nn.Sequential(
    nn.Dropout(p=0.6),
    nn.Linear(eff_model.classifier.in_features, num_class))

In [ ]:
print(eff_model)

# Hierarchical-Aware Contrastive Loss + Custom MLP

input_dim = Feature size of CNN backbone (eff_net = 1536)
- so can change the input dim as can differ for each cnn model
- input_dim = backbone_output_dim

In [ ]:
class HierarchialMLP(nn.Module):
  def __init__(self, input_dim, hidden_dims=[512, 256,128], output_dim=num_class):
    super().__init__()
    layers=[]
    dims=[input_dim] + hidden_dims
    for i in range(len(dims) - 1):
      layers.append(nn.Linear(dims[i], dims[i+1]))
      layers.append(nn.BatchNorm1d(dims[i+1])) # Normalise before activation
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(0.2)) # Regularisation
    layers.append(nn.Linear(dims[-1], output_dim)) # Final projection
    self.encoder = nn.Sequential(*layers)

  def forward(self, x):
      return self.encoder(x)

In [ ]:
models_to_train = {
    "ResNet18": res_model18,
    "EfficientNet": eff_model,
    "ResNet50": res_model50
}



Batch Size	Effect on BatchNorm
- 32	Noisy estimates, unstable training
- 32–64	Stable normalization, good convergence
- 64	Even smoother, but may require more memory

# Contrastive Learning Own Model Inspired by MoCo

- could look at using vision transformer/ resnet as encoder? (Feature Extractor)

- Alter queue size = reduce memory uptake

- Fusion model with metadata

-Momentum change

- Look at forward and then backward

- learn from previous and combine?

1️⃣ Train MoCo Forward as usual → Learn feature embeddings using fresh query samples 2️⃣ Train MoCo Backward → Use stored historical embeddings to refine representations 3️⃣ Fuse their learned representations → Use feature averaging or attention-based fusion

In [ ]:
#encoder_vision1= timm.create_model("vit_base_patch16_224", pretrained=True)
#encoder_vision1.head = nn.Linear(encoder_vision1.head.in_features, num_class)
#encoder_vision2= timm.create_model("vit_base_patch16_224", pretrained=True)
#encoder_vision2.head = nn.Identity()

In [ ]:
"""encoder_vision1 = timm.create_model("efficientnet_b0", pretrained=True)
encoder_vision1.classifier = nn.Identity()

encoder_vision2 = timm.create_model("efficientnet_b0", pretrained=True)
encoder_vision2.classifier = nn.Identity()
encoder_vision2.eval()"""

In [ ]:
"""def update_momentum_encoder(encoder_vision1, encoder_vision2, momentum=0.999):
    for param_q, param_k in zip(encoder_vision1.parameters(), encoder_vision2.parameters()):
        param_k.data = param_k.data * momentum + param_q.data * (1. - momentum)"""

In [ ]:
"""def moco_forward(encoder_vision1, encoder_vision2, queue, x):
    query_features = encoder_vision1(x)
    with torch.no_grad():
        mom_features = encoder_vision2(x)
    queue = torch.cat([mom_features, queue], dim=0)[:queue.shape[0]]
    return query_features, mom_features, queue"""

In [ ]:
"""def moco_backward(encoder_q, queue, x):
    # Skip re-encoding queue; just combine it
    query_features = encoder_q(x)

    # Ensure query features are pooled
    if query_features.dim() > 2:
        query_features = F.adaptive_avg_pool2d(query_features, 1).view(query_features.size(0), -1)

    refined = torch.mean(torch.vstack([query_features, queue]), dim=0, keepdim=True)
    queue = torch.roll(queue, shifts=-query_features.size(0), dims=0)
    queue[-query_features.size(0):] = query_features.detach()

    return refined.expand(query_features.size(0), -1), queue"""


In [ ]:
""""class MoCoEffComboModel(nn.Module):
    def __init__(self, encoder_q, encoder_k, queue, embedding_dim, alpha=0.5, num_class=3):
        super().__init__()
        self.encoder_q = encoder_q
        self.encoder_k = encoder_k
        self.queue = queue
        self.alpha = alpha
        self.classifier = nn.Linear(embedding_dim, num_class)

    def forward(self, x):
        query_features, mom_features, self.queue = moco_forward(self.encoder_q, self.encoder_k, self.queue, x)
        refined_features, self.queue = moco_backward(self.encoder_q, self.queue, x)

        # 🧼 Flatten features if needed
        if query_features.dim() > 2:
            query_features = F.adaptive_avg_pool2d(query_features, 1).view(query_features.size(0), -1)
            refined_features = F.adaptive_avg_pool2d(refined_features, 1).view(refined_features.size(0), -1)

        combo = self.alpha * query_features + (1 - self.alpha) * refined_features
        return self.classifier(combo)

    def update_momentum(self, momentum=0.999):
        update_momentum_encoder(self.encoder_q, self.encoder_k, momentum)"""


In [ ]:
"""# Inspect the feature extractor's output to determine embedding size
sample_input = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    sample_output = encoder_vision1(sample_input)
embedding_dim = sample_output.shape[1]"""


In [ ]:
"""queue_size = 64 * 20  # Based on batch size and history window

queue = torch.zeros(queue_size, embedding_dim).to(device)

moco_model = MoCoEffComboModel(
    encoder_q=encoder_vision1,
    encoder_k=encoder_vision2,
    queue=queue,
    embedding_dim=embedding_dim,  # ✅ passed directly
    alpha=0.5,
    num_class=num_class
)
"""


1️⃣ MoCo Forward → Extracts new features 2️⃣ MoCo Backward → Processes historical embeddings 3️⃣ Fusion Step → Combines both representations using alpha 4️⃣ Queue Update → Ensures negative samples evolve dynamically

Need specified loss function
- NT-Xent (Normalized Temperature-scaled Cross Entropy Loss)

✅ 2️⃣ Train the MoCo Model

Use moco_combo() to compute embeddings for training data.

Apply contrastive loss during each batch update.

✅ 3️⃣ Fine-Tune on Classification

Once pretraining is complete, use the MoCo-trained embeddings as inputs to a classifier.

Train a fully connected layer or a small CNN on top of learned features.

✅ 4️⃣ Run Models on Train, Validation, and Test Sets

Your architecture supports train_set, val_set, and test_set.

Evaluate MoCo embeddings on test data after fine-tuning.

# Models running

In [ ]:
device = 'cuda' if torch.cuda.is_available()\
          else 'mps' if torch.mps.is_available()\
          else 'cpu'
print('device', device)

In [ ]:
!pip install focal-loss

In [ ]:
!pip show focal-loss

In [ ]:
# As using pytorch need class to use focal loss
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
        log_probs = F.log_softmax(inputs, dim=1)
        probs = torch.exp(log_probs)
        targets_one_hot = F.one_hot(targets, num_classes=inputs.size(1)).float()

        focal_term = (1 - probs) ** self.gamma
        loss = -targets_one_hot * focal_term * log_probs

        if self.alpha is not None:
            alpha = torch.tensor(self.alpha).to(inputs.device)
            loss = loss * alpha

        loss = loss.sum(dim=1)

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss


In [ ]:
#criteria = torch.nn.CrossEntropyLoss(label_smoothing=0.1) original attempt
# Initially without class weights seems to be overfittinhg
class_weights = [class_weights_dict[i] for i in range(num_class)]
criteria = FocalLoss(gamma=2.0, alpha=class_weights)

#Epoch

In [ ]:
from torchmetrics import Accuracy, Precision, Recall, F1Score, AUROC, ConfusionMatrix, MeanMetric

In [ ]:
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassPrecision,
    MulticlassRecall,
    MulticlassF1Score
)

In [ ]:
from tqdm import tqdm
import time

def one_train_epoch(model, optimizer):
    start_time = time.time()

    # Macro metrics
    losses = MeanMetric().to(device)
    acc = MulticlassAccuracy(num_classes=num_class, average='macro').to(device)
    precision = MulticlassPrecision(num_classes=num_class, average='macro').to(device)
    recall = MulticlassRecall(num_classes=num_class, average='macro').to(device)
    f1_macro = MulticlassF1Score(num_classes=num_class, average='macro').to(device)

    # Per-class F1
    f1_per_class = MulticlassF1Score(num_classes=num_class, average=None).to(device)

    model.train()

    all_preds= []
    all_targets=[]

    for batch_idx, (X, Y) in enumerate(tqdm(train_loader, desc="Training", leave=False)):
        X, Y = X.to(device), Y.to(device)

        optimizer.zero_grad()
        preds = model(X)
        loss = criteria(preds, Y)
        loss.backward()
        optimizer.step()


        losses.update(loss.item(), X.size(0))
        acc.update(preds, Y)
        precision.update(preds, Y)
        recall.update(preds, Y)
        f1_macro.update(preds, Y)

        all_preds.append(preds.argmax(dim=1))
        all_targets.append(Y)

    # Concatenate predictions and targets for per-class F1
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    f1_class_scores = f1_per_class(all_preds, all_targets)


    print(f"Train epoch duration: {time.time() - start_time:.2f} seconds")
    for i, score in enumerate(f1_class_scores):
        print(f"Train F1 - Class {i}: {score:.4f}")

    return (
        losses.compute().item(),
        acc.compute().item(),
        precision.compute().item(),
        recall.compute().item(),
        f1_macro.compute().item(),
        f1_class_scores.cpu().numpy()  # Optional: return for logging or plotting
    )

In [ ]:
def one_val_epoch(model):
    start_time = time.time()

    # Macro metrics
    losses = MeanMetric().to(device)
    acc = MulticlassAccuracy(num_classes=num_class, average='macro').to(device)
    precision = MulticlassPrecision(num_classes=num_class, average='macro').to(device)
    recall = MulticlassRecall(num_classes=num_class, average='macro').to(device)
    f1_macro = MulticlassF1Score(num_classes=num_class, average='macro').to(device)
    auc = MulticlassAUROC(num_classes=num_class, average='macro').to(device)

    # Per-class F1
    f1_per_class = MulticlassF1Score(num_classes=num_class, average=None).to(device)

    model.eval()
    model.to(device)

    all_preds=[]
    all_targets=[]

    with torch.no_grad():
        for X, Y in tqdm(val_loader, desc="Validating", leave=False):
            X, Y = X.to(device), Y.to(device)
            preds = model(X)
            loss = criteria(preds, Y)

            losses.update(loss.item(), X.size(0))
            acc.update(preds, Y)
            precision.update(preds, Y)
            recall.update(preds, Y)
            f1_macro.update(preds, Y)
            auc.update(preds, Y)

            all_preds.append(preds.argmax(dim=1))
            all_targets.append(Y)

    # Compute per-class F1
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    f1_class_scores = f1_per_class(all_preds, all_targets)

    print(f"Validation took {time.time() - start_time:.2f}s")
    for i, score in enumerate(f1_class_scores):
        print(f"Val F1 - Class {i}: {score:.4f}")

    return (
        losses.compute().item(),
        acc.compute().item(),
        precision.compute().item(),
        recall.compute().item(),
        f1_macro.compute().item(),
        auc.compute().item(),
        f1_class_scores.cpu().numpy()  # Optional: return for logging or plotting
    )


In [ ]:
from sklearn.metrics import classification_report

In [ ]:
def one_test_epoch(model):
    start_time = time.time()

    # Macro metrics
    losses = MeanMetric().to(device)
    acc = MulticlassAccuracy(num_classes=num_class, average='macro').to(device)
    precision = MulticlassPrecision(num_classes=num_class, average='macro').to(device)
    recall = MulticlassRecall(num_classes=num_class, average='macro').to(device)
    f1_macro = MulticlassF1Score(num_classes=num_class, average='macro').to(device)
    auc = MulticlassAUROC(num_classes=num_class, average='macro').to(device)

    # Per-class F1
    f1_per_class = MulticlassF1Score(num_classes=num_class, average=None).to(device)

    model.eval()
    model.to(device)

    true_labels_log = []
    predictions_log = []

    all_preds=[]
    all_targets=[]

    with torch.no_grad():
        for X, Y in tqdm(test_loader, desc="Testing", leave=False):
            X, Y = X.to(device), Y.to(device)
            preds = model(X)
            loss = criteria(preds, Y)

            losses.update(loss.item(), X.size(0))
            acc.update(preds, Y)
            precision.update(preds, Y)
            recall.update(preds, Y)
            f1_macro.update(preds, Y)
            auc.update(preds, Y)

            pred_labels = preds.argmax(dim=1)
            true_labels_log.extend(Y.cpu().numpy())
            predictions_log.extend(pred_labels.cpu().numpy())

            all_preds.append(pred_labels)
            all_targets.append(Y)

    # Compute per-class F1
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    f1_class_scores = f1_per_class(all_preds, all_targets)

    print(f"Testing took {time.time() - start_time:.2f}s")
    print("\nDetailed Classification Report:")
    print(classification_report(true_labels_log, predictions_log, digits=4))

    # Save full classification report
    report_dict = classification_report(true_labels_log, predictions_log, digits=4, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv(os.path.join(project_root, f"{model_name}_class_report.csv"))

    # ✅ Save per-class metrics only (excluding summary rows)
    per_class_metrics = report_df.drop(['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
    per_class_metrics.to_csv(os.path.join(project_root, f"{model_name}_per_class_metrics.csv"))

    # Optional: Print per-class F1 from torchmetrics
    print("\nTorchMetrics Per-Class F1:")
    for i, score in enumerate(f1_class_scores):
        print(f"Test F1 - Class {i}: {score:.4f}")

    return (
        losses.compute().item(),
        acc.compute().item(),
        precision.compute().item(),
        recall.compute().item(),
        f1_macro.compute().item(),
        auc.compute().item(),
        true_labels_log,
        predictions_log,
        f1_class_scores.cpu().numpy()  # Optional: return for logging or visualization
    )


In [ ]:
train_val_results = []

def run_training(model_name, model, optimizer, scheduler,
                 train_val_path, test_results_path, checkpoint_path, num_epochs=30):

    model = model.to(device)
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')
    start_epoch = 0
    patience, cooldown = 5, 0

    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = checkpoint.get("epoch", 0) + 1
        print(f"{model_name} Resuming training from epoch {start_epoch}...")
        if checkpoint.get("tag") != "final_model" and start_epoch < num_epochs:
            print("Warning: Resumed checkpoint may not reflect final model state.")

    for epoch in range(start_epoch, num_epochs):
        train_loss, train_acc, train_prec, train_rec, train_f1, train_f1_per_class = one_train_epoch(model, optimizer)
        val_loss, val_acc, val_prec, val_rec, val_f1, val_auc, val_f1_per_class = one_val_epoch(model)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Current LR: {current_lr:.6f}")

        if not all(map(torch.isfinite, [torch.tensor(train_loss).to(device),
                                        torch.tensor(val_loss).to(device)])):
            print("Detected unstable loss (NaN/Inf). Stopping early.")
            break

        metrics = {
            "Model": model_name, "Epoch": epoch,
            "Train Loss": train_loss, "Train Accuracy": train_acc,
            "Train Precision": train_prec, "Train Recall": train_rec, "Train F1": train_f1,
            "Val Loss": val_loss, "Val Accuracy": val_acc,
            "Val Precision": val_prec, "Val Recall": val_rec, "Val F1": val_f1, "Val AUC": val_auc
        }

        train_val_results.append(metrics)
        pd.DataFrame([metrics]).to_csv(train_val_path, mode='a',
                                       header=not os.path.exists(train_val_path),
                                       index=False)

        # ✅ Log per-class F1s across epochs
        f1_df = pd.DataFrame({
            "Epoch": [epoch] * num_class,
            "Class": list(range(num_class)),
            "Train F1": train_f1_per_class,
            "Val F1": val_f1_per_class
        })
        f1_df.to_csv(os.path.join(project_root, f"{model_name}_f1_per_class_by_epoch.csv"),
                     mode='a', header=not os.path.exists(os.path.join(project_root, f"{model_name}_f1_per_class_by_epoch.csv")),
                     index=False)

        print(f"Epoch {epoch+1}/{num_epochs} | Train F1: {train_f1:.4f} | "
              f"Train Loss: {train_loss:.4f} || Val F1: {val_f1:.4f} | Val Loss: {val_loss:.4f}")

        # Save periodic checkpoint
        if (epoch + 1) % 5 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'tag': 'intermediate'
            }, os.path.join(project_root, f"{model_name}_checkpoint_epoch_{epoch}.pth"))

        # Always save latest
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'tag': 'latest'
        }, checkpoint_path)

        # Track best model weights
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            cooldown = 0
        else:
            cooldown += 1
            if cooldown >= patience:
                print("Early stopping activated.")
                break

        # Clear memory
        del train_loss, train_acc, train_prec, train_rec, train_f1, train_f1_per_class
        del val_loss, val_acc, val_prec, val_rec, val_f1, val_auc, val_f1_per_class
        gc.collect()

    # Restore best weights
    model.load_state_dict(best_model_wts)

    print(f"{model_name} Training complete. Running test set...")

    test_loss, test_acc, test_prec, test_rec, test_f1, test_auc, y_true, y_pred, test_f1_per_class = one_test_epoch(model)

    test_row = {
        "Model": model_name,
        "Test Loss": test_loss, "Test Accuracy": test_acc,
        "Test Precision": test_prec, "Test Recall": test_rec,
        "Test F1": test_f1, "Test AUC": test_auc
    }

    pd.DataFrame([test_row]).to_csv(test_results_path, mode='a',
                                    header=not os.path.exists(test_results_path),
                                    index=False)

    # Save prediction log
    pred_df = pd.DataFrame({
        "actual": y_true,
        "predicted": y_pred
    })
    pred_df.to_csv(os.path.join(project_root, f"{model_name}_predictions.csv"))

    # Save per-class metrics from classification report
    report_dict = classification_report(y_true, y_pred, digits=4, output_dict=True)
    per_class_metrics = pd.DataFrame(report_dict).transpose().drop(['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
    per_class_metrics.to_csv(os.path.join(project_root, f"{model_name}_per_class_metrics.csv"))

    # ✅ Save torchmetrics per-class F1 from test
    test_f1_df = pd.DataFrame({
        "Class": list(range(num_class)),
        "Test F1": test_f1_per_class
    })
    test_f1_df.to_csv(os.path.join(project_root, f"{model_name}_test_f1_per_class.csv"), index=False)

    print("Test results saved, predictions log saved, and per-class metrics exported.")


# Dry Run

In [ ]:
dry_run= False # To False when wanting to do full training

In [ ]:
if dry_run:
    print("[Dry Run Mode] Using small data loaders and fewer epochs.")

    train_subset = torch.utils.data.Subset(train_set, range(64))
    val_subset = torch.utils.data.Subset(val_set, range(32))
    test_subset = torch.utils.data.Subset(test_set, range(32))

    train_loader = DataLoader(train_subset, batch_size=8)
    val_loader = DataLoader(val_subset, batch_size=8)
    test_loader = DataLoader(test_subset, batch_size=8)


In [ ]:
if dry_run:
    print(f"[Dry Run Mode] Overriding num_epochs for {models_to_train}")
    num_epochs = 1


#Models to train

In [ ]:
models_to_train = [
    ("EfficientNet", eff_model),
    ("ResNet18", res_model18)
    #("ResNet50", res_model50)
]


In [ ]:
# Maybe add scheduler
from torch.optim.lr_scheduler import CosineAnnealingLR

In [ ]:
for model_name, model in models_to_train:
    # Define paths early so we can check status
    checkpoint_path = os.path.join(project_root, f"{model_name}_checkpoint_latest.pth")
    train_val_path = os.path.join(project_root, f"{model_name}_train_val_log.csv")
    test_results_path = os.path.join(project_root, f"{model_name}_test_results.csv")

    # Skip if already trained and evaluated
    if os.path.exists(test_results_path):
        print(f"[✓] {model_name} already trained and tested. Skipping.\n")
        continue

    print(f"Starting training for {model_name}...\n")

    # Clear memory before model allocation to prevent reconnect crashes
    torch.cuda.empty_cache()
    gc.collect()

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)

    run_training(model_name, model, optimizer,scheduler,
                 train_val_path, test_results_path, checkpoint_path)

    # 🧼 Clear memory after training completes
    del model, optimizer
    torch.cuda.empty_cache()
    gc.collect()


# MLP

In [ ]:
def extract_features(model, loader):
    model.eval()
    features, labels = [], []
    with torch.no_grad():
        for X, Y in loader:
            X = X.to(device)
            out = model.forward(X)
            out = out.view(out.size(0), -1)  # Flatten
            features.append(out.cpu())
            labels.append(Y)
    return torch.cat(features), torch.cat(labels)


run for train and val loader

NEED TO GET FEATURE DIM for different models

In [ ]:
def get_feature_dim(model, input_size=(1, 3, 384, 384)):
    model.eval()
    dummy_input = torch.randn(*input_size).to(device)
    with torch.no_grad():
        if hasattr(model, 'forward_features'):  # timm models
            features = model.forward_features(dummy_input)
        else:
            features = nn.Sequential(*list(model.children())[:-1])(dummy_input)
    return features.view(features.size(0), -1).size(1)


In [ ]:
feature_dim = get_feature_dim(res_model18)


In [ ]:
mlp = HierarchialMLP(input_dim=feature_dim, output_dim=num_class).to(device)
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)


In [ ]:
checkpoint = torch.load(os.path.join(project_root, "EfficientNet_checkpoint_latest.pth"), map_location=device)
eff_model.load_state_dict(checkpoint["model_state_dict"])
eff_model = eff_model.to(device)
eff_model.eval()

In [ ]:
# Assuming you have train_loader and val_loader already defined
train_feats, train_labels = extract_features(eff_model, train_loader)
val_feats, val_labels = extract_features(eff_model, val_loader)


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(train_feats, train_labels)
val_dataset = TensorDataset(val_feats, val_labels)

train_loader_mlp = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader_mlp = DataLoader(val_dataset, batch_size=32)


In [ ]:
def train_mlp(model, optimizer, train_loader, val_loader,
              num_epochs=30, model_name="mlp['EffNet']",
              log_path="mlp_effNet_metrics_log.csv"):

    criterion = FocalLoss(gamma=2.0, alpha=class_weights)
    best_val_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        model.train()
        train_loss = MeanMetric().to(device)
        acc = MulticlassAccuracy(num_classes=num_class, average='macro').to(device)
        precision = MulticlassPrecision(num_classes=num_class, average='macro').to(device)
        recall = MulticlassRecall(num_classes=num_class, average='macro').to(device)
        f1_macro = MulticlassF1Score(num_classes=num_class, average='macro').to(device)
        auc = MulticlassAUROC(num_classes=num_class, average='macro').to(device)

        for X, Y in train_loader:
            X, Y = X.to(device), Y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, Y)
            loss.backward()
            optimizer.step()

            train_loss.update(loss.item(), X.size(0))
            acc.update(out, Y)
            precision.update(out, Y)
            recall.update(out, Y)
            f1_macro.update(out, Y)
            auc.update(out, Y)

        # Validation
        val_loss = MeanMetric().to(device)
        val_acc = MulticlassAccuracy(num_classes=num_class, average='macro').to(device)
        val_precision = MulticlassPrecision(num_classes=num_class, average='macro').to(device)
        val_recall = MulticlassRecall(num_classes=num_class, average='macro').to(device)
        val_f1_macro = MulticlassF1Score(num_classes=num_class, average='macro').to(device)
        val_auc = MulticlassAUROC(num_classes=num_class, average='macro').to(device)
        f1_per_class = MulticlassF1Score(num_classes=num_class, average=None).to(device)

        all_preds, all_targets = [], []

        model.eval()
        with torch.no_grad():
            for X, Y in val_loader:
                X, Y = X.to(device), Y.to(device)
                out = model(X)
                loss = criterion(out, Y)

                val_loss.update(loss.item(), X.size(0))
                val_acc.update(out, Y)
                val_precision.update(out, Y)
                val_recall.update(out, Y)
                val_f1_macro.update(out, Y)
                val_auc.update(out, Y)

                all_preds.append(out.argmax(dim=1))
                all_targets.append(Y)

        all_preds = torch.cat(all_preds)
        all_targets = torch.cat(all_targets)
        f1_class_scores = f1_per_class(all_preds, all_targets)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss.compute():.4f} | Acc: {acc.compute():.4f} | "
              f"Prec: {precision.compute():.4f} | Rec: {recall.compute():.4f} | F1: {f1_macro.compute():.4f} || "
              f"Val Loss: {val_loss.compute():.4f} | Acc: {val_acc.compute():.4f} | Prec: {val_precision.compute():.4f} | "
              f"Rec: {val_recall.compute():.4f} | F1: {val_f1_macro.compute():.4f} | AUC: {val_auc.compute():.4f}")

        for i, score in enumerate(f1_class_scores):
            print(f"Val F1 - Class {i}: {score:.4f}")

        # Save metrics to CSV
        metrics = {
            "Model": model_name,
            "Epoch": epoch + 1,
            "Train Loss": train_loss.compute().item(),
            "Train Accuracy": acc.compute().item(),
            "Train Precision": precision.compute().item(),
            "Train Recall": recall.compute().item(),
            "Train F1": f1_macro.compute().item(),
            "Train AUC": auc.compute().item(),
            "Val Loss": val_loss.compute().item(),
            "Val Accuracy": val_acc.compute().item(),
            "Val Precision": val_precision.compute().item(),
            "Val Recall": val_recall.compute().item(),
            "Val F1": val_f1_macro.compute().item(),
            "Val AUC": val_auc.compute().item()
        }

        # Add per-class F1 scores
        for i, score in enumerate(f1_class_scores):
            metrics[f"Val F1 - Class {i}"] = score.item()

        pd.DataFrame([metrics]).to_csv(log_path, mode='a',
                                       header=not os.path.exists(log_path),
                                       index=False)

        if val_loss.compute().item() < best_val_loss:
            best_val_loss = val_loss.compute().item()
            best_model_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_wts)
    print("MLP training complete. Metrics saved to:", log_path)


In [ ]:
test_feats, test_labels = extract_features(eff_model, test_loader)
test_dataset = TensorDataset(test_feats, test_labels)
test_loader_mlp = DataLoader(test_dataset, batch_size=64)

In [ ]:
def evaluate_mlp(model, test_loader, model_name="MLP_EffNet"):
    start_time = time.time()

    # Metric trackers
    losses = MeanMetric().to(device)
    acc = MulticlassAccuracy(num_classes=num_class, average='macro').to(device)
    precision = MulticlassPrecision(num_classes=num_class, average='macro').to(device)
    recall = MulticlassRecall(num_classes=num_class, average='macro').to(device)
    f1_macro = MulticlassF1Score(num_classes=num_class, average='macro').to(device)
    auc = MulticlassAUROC(num_classes=num_class, average='macro').to(device)
    f1_per_class = MulticlassF1Score(num_classes=num_class, average=None).to(device)

    model.eval()
    model.to(device)

    true_labels_log = []
    predictions_log = []
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X, Y in tqdm(test_loader, desc="Testing MLP", leave=False):
            X, Y = X.to(device), Y.to(device)
            out = model(X)
            loss = criteria(out, Y)

            losses.update(loss.item(), X.size(0))
            acc.update(out, Y)
            precision.update(out, Y)
            recall.update(out, Y)
            f1_macro.update(out, Y)
            auc.update(out, Y)

            pred_labels = out.argmax(dim=1)
            true_labels_log.extend(Y.cpu().numpy())
            predictions_log.extend(pred_labels.cpu().numpy())

            all_preds.append(pred_labels)
            all_targets.append(Y)

    # Compute per-class F1
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    f1_class_scores = f1_per_class(all_preds, all_targets)

    print(f"Testing took {time.time() - start_time:.2f}s")
    print("\nDetailed Classification Report:")
    print(classification_report(true_labels_log, predictions_log, digits=4))

    # Save full classification report
    report_dict = classification_report(true_labels_log, predictions_log, digits=4, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv(os.path.join(project_root, f"{model_name}_class_report.csv"))

    # Save per-class metrics only
    per_class_metrics = report_df.drop(['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
    per_class_metrics.to_csv(os.path.join(project_root, f"{model_name}_per_class_metrics.csv"))

    print("\nTorchMetrics Per-Class F1:")
    for i, score in enumerate(f1_class_scores):
        print(f"Test F1 - Class {i}: {score:.4f}")

    return (
        losses.compute().item(),
        acc.compute().item(),
        precision.compute().item(),
        recall.compute().item(),
        f1_macro.compute().item(),
        auc.compute().item(),
        true_labels_log,
        predictions_log,
        f1_class_scores.cpu().numpy()
    )


In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score

def evaluate_mlp(model, test_loader, model_name="mlp['EffNet']",
                 results_path="mlp_test_results.csv",
                 conf_matrix_path="mlp_effNet_confusion_matrix.csv"):

    model.eval()
    y_true, y_pred = [], []
    test_loss = 0.0
    criterion = FocalLoss(gamma=2.0, alpha=class_weights)

    with torch.no_grad():
        for X, Y in test_loader:
            X, Y = X.to(device), Y.to(device)
            out = model(X)
            loss = criterion(out, Y)
            test_loss += loss.item() * X.size(0)
            preds = out.argmax(1).cpu()
            y_true.extend(Y.cpu().numpy())
            y_pred.extend(preds.numpy())

    # Compute metrics
    test_acc = accuracy_score(y_true, y_pred)
    test_f1 = f1_score(y_true, y_pred, average='weighted')
    test_prec = precision_score(y_true, y_pred, average='weighted')
    test_rec = recall_score(y_true, y_pred, average='weighted')
    test_loss /= len(test_loader.dataset)

    # Save summary row
    test_row = {
        "Model": model_name,
        "Test Loss": test_loss,
        "Test Accuracy": test_acc,
        "Test Precision": test_prec,
        "Test Recall": test_rec,
        "Test F1": test_f1
    }

    pd.DataFrame([test_row]).to_csv(results_path, mode='a',
                                    header=not os.path.exists(results_path),
                                    index=False)

    # Save confusion matrix
    conf_matrix = confusion_matrix(y_true, y_pred)
    conf_df = pd.DataFrame(conf_matrix)
    conf_df.to_csv(conf_matrix_path, index=False)

    print(f"Test results saved to {results_path}")
    print(f"Confusion matrix saved to {conf_matrix_path}")


In [ ]:
feature_dim = train_feats.shape[1]  # Assuming train_feats is [N, D]
mlp = HierarchialMLP(input_dim=feature_dim, output_dim=num_class).to(device)

optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)


In [ ]:
train_mlp(mlp, optimizer, train_loader_mlp, val_loader_mlp, num_epochs=30)


In [ ]:
evaluate_mlp(mlp, test_loader_mlp)


# Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
def load_model(model_name, model_fn, num_class, checkpoint_path):
    # Initialize model with correct number of classes
    model = model_fn(num_classes=num_class)

    # Replace head manually if needed (for models that default to 1000 classes)
    if hasattr(model, 'head'):
        model.head = nn.Linear(model.head.in_features, num_class)
    elif hasattr(model, 'fc'):
        model.fc = nn.Linear(model.fc.in_features, num_class)
    elif hasattr(model, 'classifier'):
        model.classifier = nn.Linear(model.classifier.in_features, num_class)

    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint.get('model_state_dict', checkpoint)

    # Load state dict non-strictly to bypass head mismatch
    model.load_state_dict(state_dict, strict=False)

    return model.to(device).eval()


In [ ]:
def get_confusion_matrix(model, loader, class_names, title):
    all_preds, all_targets = [], []

    with torch.no_grad():
        for X, Y in loader:
            X, Y = X.to(device), Y.to(device)
            outputs = model(X)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(Y.cpu().numpy())

    cm = confusion_matrix(all_targets, all_preds)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)

    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(f"{title.replace(' ', '_')}.png", dpi=300)
    plt.show()


In [ ]:
model_configs = [
    ("ResNet18", lambda num_classes: torchvision.models.resnet18(weights=None, num_classes=num_classes), "/content/drive/MyDrive/Thesis/DaBIGone/ResNet18_checkpoint_latest.pth"),
    ("EfficientNet", lambda num_classes: timm.create_model("efficientnet_b3", pretrained=False, num_classes=num_classes), "/content/drive/MyDrive/Thesis/DaBIGone/EfficientNet_checkpoint_latest.pth"),
]

In [ ]:
for model_name, model_fn, ckpt in model_configs:
    print(f"Evaluating {model_name}...")
    model = load_model(model_name, model_fn, num_class, ckpt)
    get_confusion_matrix(model, test_loader, classes, title=f"{model_name} Confusion Matrix")

# Results

In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
res_preds = pd.read_csv('/content/drive/MyDrive/Thesis/DaBIGone/ResNet18_predictions.csv')
eff_preds = pd.read_csv('/content/drive/MyDrive/Thesis/DaBIGone/EfficientNet_predictions.csv')

In [ ]:
classes

In [ ]:
res_preds.sample()

In [ ]:
res_preds['actual'] = res_preds['actual'].replace({0:'AK', 1:'BCC', 2:'BKL', 3:'DF', 4:'MEL', 5:'NV', 6:'SCC', 7:'VASC'})
res_preds['predicted'] = res_preds['predicted'].replace({0:'AK', 1:'BCC', 2:'BKL', 3:'DF', 4:'MEL', 5:'NV', 6:'SCC', 7:'VASC'})

In [ ]:
eff_preds['actual'] = eff_preds['actual'].replace({0:'AK', 1:'BCC', 2:'BKL', 3:'DF', 4:'MEL', 5:'NV', 6:'SCC', 7:'VASC'})
eff_preds['predicted'] = eff_preds['predicted'].replace({0:'AK', 1:'BCC', 2:'BKL', 3:'DF', 4:'MEL', 5:'NV', 6:'SCC', 7:'VASC'})

In [ ]:
def analyze_semantic_misclassifications(results_df):
    cancerous = {'BCC', 'MEL', 'SCC'}
    precancerous = {'AK'}
    non_cancerous = {'BKL', 'DF', 'NV', 'VASC'}

    semantic_misclassifications = []

    for idx, row in results_df.iterrows():
        actual = row['actual']
        predicted = row['predicted']

        if actual != predicted:
            if actual in cancerous:
                if predicted in cancerous:
                    category = 'Cancerous → Cancerous'
                elif predicted in precancerous:
                    category = 'Cancerous → Precancerous'
                else:
                    category = 'Cancerous → Non-cancerous'
                semantic_misclassifications.append({
                    'index': idx,
                    'actual': actual,
                    'predicted': predicted,
                    'category': category
                })

    return pd.DataFrame(semantic_misclassifications)


In [ ]:
mis_res_preds = analyze_semantic_misclassifications(res_preds)

In [ ]:
mis_eff_preds = analyze_semantic_misclassifications(eff_preds)

In [ ]:
mis_res_preds.head()

In [ ]:
mis_eff_preds.head()

In [ ]:
sns.countplot(y='category', hue='category',data=mis_res_preds)
plt.title('Semantic Misclassification Breakdown for Cancerous Cases (ResNet18)',weight='bold')

In [ ]:
sns.countplot(y='category', hue='category',data=mis_eff_preds)
plt.title('Semantic Misclassification Breakdown for Cancerous Cases (Efficient Net)',weight='bold')

# FINAL RESULTS

In [ ]:
res_per_class = pd.read_csv('/content/drive/MyDrive/Thesis/DaBIGone/ResNet18_f1_per_class_by_epoch.csv')
eff_per_class = pd.read_csv('/content/drive/MyDrive/Thesis/DaBIGone/EfficientNet_f1_per_class_by_epoch.csv')

In [ ]:
res_per_class.sample()

In [ ]:
res_per_class['Class'] = res_per_class['Class'].replace({0:'AK', 1:'BCC', 2:'BKL', 3:'DF', 4:'MEL', 5:'NV', 6:'SCC', 7:'VASC'})
eff_per_class['Class'] = eff_per_class['Class'].replace({0:'AK', 1:'BCC', 2:'BKL', 3:'DF', 4:'MEL', 5:'NV', 6:'SCC', 7:'VASC'})

In [ ]:
res_per_class.sample()

In [ ]:
sns.set(style='whitegrid')
sns.lineplot(x='Epoch', y='Train F1', hue='Class', data=res_per_class)
plt.legend(
    title='Class Label',
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    frameon=True,
    framealpha=0.9,
    edgecolor='black',
    fontsize=12,
    title_fontsize=13
)
plt.title('Train F1 Score Per Class Across Epochs (ResNet18)', weight='bold')
plt.tight_layout()

In [ ]:
sns.set(style='whitegrid')
sns.lineplot(x='Epoch', y='Val F1', hue='Class', data=res_per_class)
plt.legend(
    title='Class Label',
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    frameon=True,
    framealpha=0.9,
    edgecolor='black',
    fontsize=12,
    title_fontsize=13
)
plt.title('Validation F1 Score Per Class Across Epochs (ResNet18)', weight='bold')
plt.tight_layout()

In [ ]:
sns.set(style='whitegrid')
sns.lineplot(x='Epoch', y='Train F1', hue='Class', data=eff_per_class)
plt.legend(
    title='Class Label',
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    frameon=True,
    framealpha=0.9,
    edgecolor='black',
    fontsize=12,
    title_fontsize=13
)
plt.title('Train F1 Score Per Class Across Epochs (Efficient Net)', weight='bold')
plt.tight_layout()

In [ ]:
sns.set(style='whitegrid')
sns.lineplot(x='Epoch', y='Val F1', hue='Class', data=eff_per_class)
plt.legend(
    title='Class Label',
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    frameon=True,
    framealpha=0.9,
    edgecolor='black',
    fontsize=12,
    title_fontsize=13
)
plt.title('Validation F1 Score Per Class Across Epochs (Efficient Net)', weight='bold')
plt.tight_layout()

In [ ]:
res_class_f1 = pd.read_csv('/content/drive/MyDrive/Thesis/DaBIGone/ResNet18_test_f1_per_class.csv')
eff_class_f1 = pd.read_csv('/content/drive/MyDrive/Thesis/DaBIGone/EfficientNet_test_f1_per_class.csv')

In [ ]:
res_class_f1

In [ ]:
res_class_f1['Class'] = res_class_f1['Class'].replace({0:'AK', 1:'BCC', 2:'BKL', 3:'DF', 4:'MEL', 5:'NV', 6:'SCC', 7:'VASC'})
eff_class_f1['Class'] = eff_class_f1['Class'].replace({0:'AK', 1:'BCC', 2:'BKL', 3:'DF', 4:'MEL', 5:'NV', 6:'SCC', 7:'VASC'})


In [ ]:
res_class_f1

In [ ]:
sns.barplot(x='Class', y='Test F1', hue='Class', data=res_class_f1)
plt.title('Comparison of Test F1 Scores by Lesion Type (ResNet18)',weight='bold')

In [ ]:
# Create the barplot without hue to avoid duplicate bars
ax = sns.barplot(x='Class', y='Test F1', data=eff_class_f1, hue='Class')

# Add value labels above each bar
for bar in ax.patches:
    x = bar.get_x() + bar.get_width() / 2
    y = bar.get_height()
    label = f"{y:.2f}"
    ax.text(x, y + 0.001, label, ha='center', va='bottom', fontsize=10)

# Add title and axis labels
plt.title('Comparison of Test F1 Scores by Lesion Type (Efficient Net)', fontsize=14, weight='bold')
plt.xlabel('Lesion Class', fontsize=12)
plt.ylabel('Test F1', fontsize=12)

# Ensure layout
plt.tight_layout()

In [ ]:

# Define semantic categories
cancerous = ['BCC', 'MEL', 'SCC']
non_cancerous = ['BKL', 'DF', 'NV', 'VASC']
precancerous = ['AK']

# Create a custom color palette
semantic_palette = {
    'BCC': '#1f77b4',  # dark blue
    'MEL': '#1f77b4',
    'SCC': '#1f77b4',
    'AK': '#ff7f0e',   # orange for precancerous
    'BKL': '#aec7e8',  # light blue
    'DF': '#aec7e8',
    'NV': '#aec7e8',
    'VASC': '#aec7e8'
}

# Create the barplot
ax = sns.barplot(x='Class', y='Test F1', data=res_class_f1, palette=semantic_palette)

# Add value labels above each bar
for bar in ax.patches:
    x = bar.get_x() + bar.get_width() / 2
    y = bar.get_height()
    label = f"{y:.2f}"
    ax.text(x, y + 0.001, label, ha='center', va='bottom', fontsize=10)

# Add title and axis labels
plt.title('Comparison of Test F1 Scores by Lesion Type (ResNet18)', fontsize=14, weight='bold')
plt.xlabel('Lesion Class', fontsize=12)
plt.ylabel('Test F1 Score', fontsize=12)

# Create custom legend for semantic categories
cancerous_patch = mpatches.Patch(color='#1f77b4', label='Cancerous')
precancerous_patch = mpatches.Patch(color='#ff7f0e', label='Precancerous')
non_cancerous_patch = mpatches.Patch(color='#aec7e8', label='Non-cancerous')

plt.legend(
    handles=[cancerous_patch, precancerous_patch, non_cancerous_patch],
    title='Lesion Category',
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),  # Pushes legend outside to the right
    frameon=True,
    fontsize=10,
    title_fontsize=11
)


# Final layout adjustment
plt.tight_layout()


In [ ]:
import matplotlib.patches as mpatches

# Define semantic categories
cancerous = ['BCC', 'MEL', 'SCC']
non_cancerous = ['BKL', 'DF', 'NV', 'VASC']
precancerous = ['AK']

# Create a custom color palette
semantic_palette = {
    'BCC': '#1f77b4',  # dark blue
    'MEL': '#1f77b4',
    'SCC': '#1f77b4',
    'AK': '#ff7f0e',   # orange for precancerous
    'BKL': '#aec7e8',  # light blue
    'DF': '#aec7e8',
    'NV': '#aec7e8',
    'VASC': '#aec7e8'
}

# Create the barplot
ax = sns.barplot(x='Class', y='Test F1', data=eff_class_f1, palette=semantic_palette)

# Add value labels above each bar
for bar in ax.patches:
    x = bar.get_x() + bar.get_width() / 2
    y = bar.get_height()
    label = f"{y:.2f}"
    ax.text(x, y + 0.001, label, ha='center', va='bottom', fontsize=10)

# Add title and axis labels
plt.title('Comparison of Test F1 Scores by Lesion Type (Efficient Net)', fontsize=14, weight='bold')
plt.xlabel('Lesion Class', fontsize=12)
plt.ylabel('Test F1 Score', fontsize=12)

# Create custom legend for semantic categories
cancerous_patch = mpatches.Patch(color='#1f77b4', label='Cancerous')
precancerous_patch = mpatches.Patch(color='#ff7f0e', label='Precancerous')
non_cancerous_patch = mpatches.Patch(color='#aec7e8', label='Non-cancerous')

plt.legend(
    handles=[cancerous_patch, precancerous_patch, non_cancerous_patch],
    title='Lesion Category',
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),  # Pushes legend outside to the right
    frameon=True,
    fontsize=10,
    title_fontsize=11
)


# Final layout adjustment
plt.tight_layout()


In [ ]:
res_train = pd.read_csv('/content/drive/MyDrive/Thesis/DaBIGone/ResNet18_train_val_log.csv')
eff_train = pd.read_csv('/content/drive/MyDrive/Thesis/DaBIGone/EfficientNet_train_val_log.csv')

In [ ]:
res_train.sample()

In [ ]:
res_test = pd.read_csv('/content/drive/MyDrive/Thesis/DaBIGone/ResNet18_test_results.csv')
eff_test = pd.read_csv('/content/drive/MyDrive/Thesis/DaBIGone/EfficientNet_test_results.csv')

In [ ]:
res_test

In [ ]:
eff_test

In [ ]:
test = pd.concat([res_test, eff_test], axis=0)

In [ ]:
test

In [ ]:
# Create the barplot without hue to avoid duplicate bars
ax = sns.barplot(x='Model', y='Test Recall', data=test, hue='Model')

# Add value labels above each bar
for bar in ax.patches:
    x = bar.get_x() + bar.get_width() / 2
    y = bar.get_height()
    label = f"{y:.2f}"
    ax.text(x, y + 0.001, label, ha='center', va='bottom', fontsize=10)

# Add title and axis labels
plt.title('Comparison of Test Precision Scores For Models', fontsize=14, weight='bold')
plt.xlabel('Model Architecture', fontsize=12)
plt.ylabel('Test Recall', fontsize=12)

# Ensure layout
plt.tight_layout()

In [ ]:
# Create the barplot without hue to avoid duplicate bars
ax = sns.barplot(x='Model', y='Test AUC', data=test, hue='Model')

# Add value labels above each bar
for bar in ax.patches:
    x = bar.get_x() + bar.get_width() / 2
    y = bar.get_height()
    label = f"{y:.2f}"
    ax.text(x, y + 0.001, label, ha='center', va='bottom', fontsize=10)

# Add title and axis labels
plt.title('Comparison of Test Area Under Curve Scores For Models', fontsize=14, weight='bold')
plt.xlabel('Model Architecture', fontsize=12)
plt.ylabel('Test AUC', fontsize=12)

# Ensure layout is tight and clean
plt.tight_layout()

In [ ]:
# Create the barplot without hue to avoid duplicate bars
ax = sns.barplot(x='Model', y='Test Accuracy', data=test, hue='Model')

# Add value labels above each bar
for bar in ax.patches:
    x = bar.get_x() + bar.get_width() / 2
    y = bar.get_height()
    label = f"{y:.2f}"
    ax.text(x, y + 0.001, label, ha='center', va='bottom', fontsize=10)

# Add title and axis labels
plt.title('Comparison of Test Accuracy For Models', fontsize=14, weight='bold')
plt.xlabel('Model Architecture', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)

# Ensure layout is tight and clean
plt.tight_layout()

In [ ]:
res_test

In [ ]:
plt.title('ResNet18 Loss Across Epochs', weight='bold')
plt.ylabel('Loss')
sns.lineplot(x='Epoch', y='Train Loss', data=res_train, label='Train Loss')
sns.lineplot(x='Epoch', y='Val Loss', data=res_train, label='Val Loss')

# Extract test loss value from res_test
test_loss_value = res_test.loc[res_test['Model'] == 'ResNet18', 'Test Loss'].values[0]

# Add horizontal line for test loss
plt.axhline(y=test_loss_value, color='red', linestyle='--', label='Test Loss')

# Add legend
plt.legend()



plt.tight_layout()

In [ ]:
eff_test

In [ ]:
plt.title('Efficient Net Loss Across Epochs',weight='bold')
plt.ylabel('Loss')
sns.lineplot(x='Epoch', y='Train Loss', data=eff_train, label='Train Loss')
sns.lineplot(x='Epoch', y='Val Loss', data=eff_train, label='Val Loss')
# Extract test loss value from res_test
test_loss_value = eff_test.loc[eff_test['Model'] == 'EfficientNet', 'Test Loss'].values[0]

# Add horizontal line for test loss
plt.axhline(y=test_loss_value, color='red', linestyle='--', label='Test Loss')

# Add legend
plt.legend()



plt.tight_layout()

# Grad-CAM

In [ ]:
!pip install torchcam

In [ ]:
from torchcam.methods import GradCAM
from torchvision.transforms import ToPILImage
import torch.nn.functional as F

Pick a Target Layer for each Model

In [ ]:
def get_extractor(model_name, model):
    if model_name == "ResNet18":
        return GradCAM(model, target_layer="layer4")  # Last conv block
    elif model_name == "EfficientNet":
        return GradCAM(model, target_layer="blocks.6")  # Mid-level block with richer spatial info


In [ ]:
model_configs = [
    ("ResNet18", res_model18, "r/content/drive/MyDrive/Thesis/DaBIGone/ResNet18_checkpoint_latest.pth"),       # Replace with your actual function and path
    ("EfficientNet", eff_model, "/content/drive/MyDrive/Thesis/DaBIGone/EfficientNet_checkpoint_latest.pth")  # Replace with your actual function and path
]


In [ ]:
for name, module in eff_model.named_modules():
  if isinstance(module, nn.Conv2d):
    print(name)

In [ ]:
for name, module in vis_model.named_modules():
  print(name)

In [ ]:
from pytorch_grad_cam import GradCAMPlusPlus

cam_extractor = GradCAMPlusPlus(model=model, target_layer=target_layer)


In [ ]:
def load_model(model_name, model_fn, num_classes, ckpt_path):
    model = model_fn if isinstance(model_fn, nn.Module) else model_fn(num_classes=num_classes)

    if ckpt_path and os.path.exists(ckpt_path):
        checkpoint = torch.load(ckpt_path, map_location=device)
        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
        else:
            model.load_state_dict(checkpoint)

    model.to(device)
    model.eval()
    return model


## Grad-CAM For 1 IMAGE

In [ ]:
def visualise_cam(model_name, model, cam_extractor, image_tensor, true_label=None, classes=None):
    model.eval()
    output = model(image_tensor.unsqueeze(0))  # Shape: (1, num_classes)
    pred_class = output.argmax().item()

    # Extract CAM
    cams = cam_extractor(pred_class, output)
    cam_tensor = cams[0]  # Might be (1, 7, 7) or (7, 7)

    # Ensure CAM shape
    if cam_tensor.dim() == 2:
        cam_tensor = cam_tensor.unsqueeze(0).unsqueeze(0)
    elif cam_tensor.dim() == 3:
        cam_tensor = cam_tensor.unsqueeze(0)
    else:
        raise ValueError(f"Unexpected CAM shape: {cam_tensor.shape}")

    # Resize CAM
    cam_resized = F.interpolate(
        cam_tensor,
        size=(image_tensor.shape[1], image_tensor.shape[2]),
        mode='bilinear',
        align_corners=False
    )
    cam = cam_resized.squeeze().cpu().numpy()

    # Normalize original image
    image_np = image_tensor.permute(1, 2, 0).cpu().numpy()
    image_np = (image_np - image_np.min()) / (image_np.max() - image_np.min())

    # Generate labels
    pred_label = classes[pred_class] if classes else str(pred_class)
    true_label_str = classes[true_label] if true_label is not None and classes else "?"

    # Side-by-side plot
    plt.figure(figsize=(10, 5))

    # Original Image
    plt.subplot(1, 2, 1)
    plt.imshow(image_np)
    plt.title("Original Image")
    plt.axis("off")

    # Grad-CAM Overlay
    plt.subplot(1, 2, 2)
    plt.imshow(image_np)
    plt.imshow(cam, cmap='jet', alpha=0.5)
    plt.title(f"{model_name}: Predicted {pred_label} | True {true_label_str}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


## Grad-CAM Across Classes

In [ ]:
def get_random_image_of_class(class_idx, dataset):
    indices = [i for i, (_, label) in enumerate(dataset) if label == class_idx]
    chosen_idx = random.choice(indices)
    image, label = dataset[chosen_idx]
    return image, label

## Same Image Across Models & Classes

In [ ]:
def compare_model_cams(image_tensor, true_label, model_configs, classes):
    plt.figure(figsize=(15, 4))

    # Original image
    image_np = image_tensor.permute(1, 2, 0).cpu().detach().numpy()
    image_np = (image_np - image_np.min()) / (image_np.max() - image_np.min())

    plt.subplot(1, len(model_configs) + 1, 1)
    plt.imshow(image_np)
    plt.title("Original")
    plt.axis("off")

    # Loop through models
    for i, (model_name, model_fn, ckpt_path) in enumerate(model_configs, start=2):
        model = load_model(model_name, model_fn, len(classes), ckpt_path)
        cam_extractor = get_extractor(model_name, model)

        output = model(image_tensor.unsqueeze(0))
        pred_class = output.argmax().item()
        pred_label = classes[pred_class]
        true_label_str = classes[true_label]

        cams = cam_extractor(pred_class, output)
        cam_tensor = cams[0]

        if cam_tensor.dim() == 2:
            cam_tensor = cam_tensor.unsqueeze(0).unsqueeze(0)
        elif cam_tensor.dim() == 3:
            cam_tensor = cam_tensor.unsqueeze(0)

        cam_resized = F.interpolate(cam_tensor, size=(image_tensor.shape[1], image_tensor.shape[2]), mode='bilinear', align_corners=False)
        cam = cam_resized.squeeze().cpu().numpy()

        plt.subplot(1, len(model_configs) + 1, i)
        plt.imshow(image_np)
        plt.imshow(cam, cmap='jet', alpha=0.5)
        plt.title(f"{model_name}\nPredicted: {pred_label}\nTrue: {true_label_str}")
        plt.axis("off")


    plt.tight_layout()
    plt.show()


In [ ]:
def get_image_by_prediction(dataset, model, class_idx, correct=True):
    model.eval()
    candidates = []

    for i, (img, label) in enumerate(dataset):
        if label != class_idx:
            continue
        img_input = img.unsqueeze(0).to(device)
        pred = model(img_input).argmax(dim=1).item()
        if (pred == class_idx) == correct:
            candidates.append((img, label))

    if not candidates:
        print(f"No {'correct' if correct else 'incorrect'} predictions found for class {class_idx}")
        return None, None

    return random.choice(candidates)


In [ ]:
classes

In [ ]:
for class_name in ["AK", "BCC", "BKL", "DF", "MEL", "NV", "SCC", "VASC"]:
    class_idx = classes.index(class_name)
    print(f"\n🔍 Class: {class_name}")

    for model_name, model in [("EfficientNet", eff_model), ("ResNet18", res_model18)]:
        for correct in [True, False]:
            image_tensor, label = get_image_by_prediction(test_loader.dataset, model, class_idx, correct=correct)

            if image_tensor is not None:
                image_tensor = F.interpolate(image_tensor.unsqueeze(0), size=(384, 384), mode='bilinear', align_corners=False).squeeze(0)
                image_tensor = image_tensor.to(device)
                image_tensor.requires_grad_()

                status = "Correct" if correct else "Incorrect"
                print(f"{status} prediction by {model_name}")
                compare_model_cams(image_tensor, label, model_configs, classes)


### AK

In [ ]:
target_class_name = 'AK'
class_idx = classes.index(target_class_name)

In [ ]:
image_tensor, label = get_random_image_of_class(class_idx, test_loader.dataset)
compare_model_cams(image_tensor, label, model_configs, classes)

In [ ]:
target_class_name = 'AK'
class_idx = classes.index(target_class_name)

# Correct prediction
image_tensor, label = get_image_by_prediction(test_loader.dataset, eff_model, class_idx, correct=True)
if image_tensor is not None:
    image_tensor = image_tensor.to(device)
    image_tensor.requires_grad_()  # Enable gradient tracking
    compare_model_cams(image_tensor, label, model_configs, classes)

# Incorrect prediction
image_tensor, label = get_image_by_prediction(test_loader.dataset, eff_model, class_idx, correct=False)
if image_tensor is not None:
    image_tensor = image_tensor.to(device)
    image_tensor.requires_grad_()  # Enable gradient tracking
    compare_model_cams(image_tensor, label, model_configs, classes)


In [ ]:
import torch.nn.functional as F
target_class_name = 'AK'
class_idx = classes.index(target_class_name)

# Correct prediction by EfficientNet
image_tensor, label = get_image_by_prediction(test_loader.dataset, eff_model, class_idx, correct=True)
if image_tensor is not None:
    image_tensor = F.interpolate(image_tensor.unsqueeze(0), size=(384, 384), mode='bilinear', align_corners=False).squeeze(0)
    image_tensor = image_tensor.to(device)
    image_tensor.requires_grad_()
    print("✓ Correct prediction by EfficientNet")
    compare_model_cams(image_tensor, label, model_configs, classes)

# Incorrect prediction by EfficientNet
image_tensor, label = get_image_by_prediction(test_loader.dataset, eff_model, class_idx, correct=False)
if image_tensor is not None:
    image_tensor = F.interpolate(image_tensor.unsqueeze(0), size=(384, 384), mode='bilinear', align_corners=False).squeeze(0)
    image_tensor = image_tensor.to(device)
    image_tensor.requires_grad_()
    print("✗ Incorrect prediction by EfficientNet")
    compare_model_cams(image_tensor, label, model_configs, classes)

# Correct prediction by ResNet18
image_tensor, label = get_image_by_prediction(test_loader.dataset, res_model18, class_idx, correct=True)
if image_tensor is not None:
    image_tensor = F.interpolate(image_tensor.unsqueeze(0), size=(384, 384), mode='bilinear', align_corners=False).squeeze(0)
    image_tensor = image_tensor.to(device)
    image_tensor.requires_grad_()
    print("✗ Incorrect prediction by ResNet18")
    compare_model_cams(image_tensor, label, model_configs, classes)

# Incorrect prediction by ResNet18
image_tensor, label = get_image_by_prediction(test_loader.dataset, res_model18, class_idx, correct=False)
if image_tensor is not None:
    image_tensor = F.interpolate(image_tensor.unsqueeze(0), size=(384, 384), mode='bilinear', align_corners=False).squeeze(0)
    image_tensor = image_tensor.to(device)
    image_tensor.requires_grad_()
    print("✗ Incorrect prediction by ResNet18")
    compare_model_cams(image_tensor, label, model_configs, classes)


### BCC

In [ ]:
target_class_name = 'BCC'
class_idx = classes.index(target_class_name)

In [ ]:
image_tensor, label = get_random_image_of_class(class_idx, test_loader.dataset)
compare_model_cams(image_tensor, label, model_configs, classes)

### BKL

In [ ]:
target_class_name = 'BKL'
class_idx = classes.index(target_class_name)

In [ ]:
image_tensor, label = get_random_image_of_class(class_idx, test_loader.dataset)
compare_model_cams(image_tensor, label, model_configs, classes)

### DF

In [ ]:
target_class_name = 'DF'
class_idx = classes.index(target_class_name)

In [ ]:
image_tensor, label = get_random_image_of_class(class_idx, test_loader.dataset)
compare_model_cams(image_tensor, label, model_configs, classes)

### MEL

In [ ]:
target_class_name = 'MEL'
class_idx = classes.index(target_class_name)

In [ ]:
image_tensor, label = get_random_image_of_class(class_idx, test_loader.dataset)
compare_model_cams(image_tensor, label, model_configs, classes)

### NV

In [ ]:
target_class_name = 'NV'
class_idx = classes.index(target_class_name)

In [ ]:
image_tensor, label = get_random_image_of_class(class_idx, test_loader.dataset)
compare_model_cams(image_tensor, label, model_configs, classes)

### SCC

In [ ]:
target_class_name = 'SCC'
class_idx = classes.index(target_class_name)

In [ ]:
image_tensor, label = get_random_image_of_class(class_idx, test_loader.dataset)
compare_model_cams(image_tensor, label, model_configs, classes)

### VASC

In [ ]:
target_class_name = 'VASC'
class_idx = classes.index(target_class_name)

In [ ]:
image_tensor, label = get_random_image_of_class(class_idx, test_loader.dataset)
compare_model_cams(image_tensor, label, model_configs, classes)

# Grad-CAM ++

In [ ]:
!pip install -q "grad-cam"


In [ ]:
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image


In [ ]:
def load_model(model_name, model_fn, num_classes, ckpt_path):
    model = model_fn if isinstance(model_fn, nn.Module) else model_fn(num_classes=num_classes)

    if ckpt_path and os.path.exists(ckpt_path):
        checkpoint = torch.load(ckpt_path, map_location=device)
        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
        else:
            model.load_state_dict(checkpoint)

    model.to(device)
    model.eval()
    return model


In [ ]:
def get_extractor(model_name, model):
    if model_name == "ResNet18":
        return GradCAMPlusPlus(model, target_layers=[model.layer4])
    elif model_name == "EfficientNet":
        return GradCAMPlusPlus(model, target_layers=[model.blocks[5]])  # or blocks[5][1]


In [ ]:
def compare_model_cams(image_tensor, true_label, model_configs, classes, class_name=None, status=None, save_fig=False):
    num_panels = len(model_configs) + 1
    plt.figure(figsize=(3.5 * num_panels, 3))  # Dynamically scale width based on number of panels

    # Normalize original image
    image_np = image_tensor.permute(1, 2, 0).cpu().detach().numpy()
    image_np = (image_np - image_np.min()) / (image_np.max() - image_np.min())

    # Original image panel
    plt.subplot(1, num_panels, 1)
    plt.imshow(image_np)
    plt.title(f"Original Class: {class_name}", fontsize=10)
    plt.axis("off")

    # Loop through models
    for i, (model_name, model_fn, ckpt_path) in enumerate(model_configs, start=2):
        model = load_model(model_name, model_fn, len(classes), ckpt_path)
        cam_extractor = get_extractor(model_name, model)

        output = model(image_tensor.unsqueeze(0))
        pred_class = output.argmax().item()
        pred_label = classes[pred_class]
        true_label_str = classes[true_label]
        confidence = torch.softmax(output, dim=1)[0, pred_class].item()

        cams = cam_extractor(image_tensor.unsqueeze(0), targets=[ClassifierOutputTarget(pred_class)])
        cam = cams[0]
        cam = (cam - cam.min()) / (cam.max() - cam.min())

        plt.subplot(1, num_panels, i)
        plt.imshow(image_np)
        plt.imshow(cam, cmap='jet', alpha=0.5)
        plt.title(f"{model_name}\nPredicted: {pred_label} \n(Confidence in Prediction: {confidence:.2f})\nTrue: {true_label_str}", fontsize=10)
        plt.axis("off")

    # Adjust spacing
    plt.subplots_adjust(wspace=0.05, hspace=0.05)  # Reduce horizontal and vertical spacing
    plt.tight_layout(pad=0.5)

    # Save figure if requested
    if save_fig and class_name and status:
        plt.savefig(f"GradCAM++_{class_name}_{status}.png", dpi=300, bbox_inches='tight', pad_inches=0.1)

    plt.show()


In [ ]:
def get_joint_prediction_case(dataset, class_idx, eff_model, res_model):
    cases = {
        "both_correct": None,
        "eff_correct_res_wrong": None,
        "res_correct_eff_wrong": None,
        "both_wrong": None
    }

    for img, label in dataset:
        if label != class_idx:
            continue

        img_input = F.interpolate(img.unsqueeze(0), size=(384, 384), mode='bilinear', align_corners=False).squeeze(0)
        img_input = img_input.to(device)
        img_input.requires_grad_()

        eff_pred = eff_model(img_input.unsqueeze(0)).argmax(dim=1).item()
        res_pred = res_model(img_input.unsqueeze(0)).argmax(dim=1).item()

        if eff_pred == label and res_pred == label and cases["both_correct"] is None:
            cases["both_correct"] = (img_input, label)
        elif eff_pred == label and res_pred != label and cases["eff_correct_res_wrong"] is None:
            cases["eff_correct_res_wrong"] = (img_input, label)
        elif eff_pred != label and res_pred == label and cases["res_correct_eff_wrong"] is None:
            cases["res_correct_eff_wrong"] = (img_input, label)
        elif eff_pred != label and res_pred != label and cases["both_wrong"] is None:
            cases["both_wrong"] = (img_input, label)

        if all(v is not None for v in cases.values()):
            break

    return cases


In [ ]:
for class_name in ["AK", "BCC", "BKL", "DF", "MEL", "NV", "SCC", "VASC"]:
    class_idx = classes.index(class_name)
    print(f"\n🔍 Class: {class_name}")

    cases = get_joint_prediction_case(test_loader.dataset, class_idx, eff_model, res_model18)

    for status, case in cases.items():
        if case is not None:
            image_tensor, label = case
            print(f" Case: {status.replace('_', ' ').title()}")

            compare_model_cams(
                image_tensor,
                label,
                model_configs,  # Includes both EfficientNet and ResNet18
                classes,
                class_name=class_name,
                status=status,
                save_fig=True
            )


In [ ]:
for class_name in ["AK", "BCC", "BKL", "DF", "MEL", "NV", "SCC", "VASC"]:
    class_idx = classes.index(class_name)
    print(f"\n🔍 Class: {class_name}")

    for correct in [True, False]:
        # Select image based on EfficientNet prediction
        image_tensor, label = get_image_by_prediction(test_loader.dataset, eff_model, class_idx, correct=correct)

        if image_tensor is not None:
            image_tensor = F.interpolate(image_tensor.unsqueeze(0), size=(384, 384), mode='bilinear', align_corners=False).squeeze(0)
            image_tensor = image_tensor.to(device)
            image_tensor.requires_grad_()

            status = "Correct" if correct else "Incorrect"
            print(f"{status} prediction by EfficientNet — comparing both models")

            # Compare both models on the same image
            compare_model_cams(
                image_tensor,
                label,
                model_configs,  # Includes both EfficientNet and ResNet18
                classes,
                class_name=class_name,
                status=status,
                save_fig=True
            )


# Ideas

1. SHAP + Metadata Fusion
Once you integrate metadata (e.g., age, sex, lesion location), use SHAP for:

Feature importance on tabular data (tree-based SHAP or DeepSHAP)

Global impact plots per class or per model

Overlaying SHAP explanations alongside CAM to show: “How much did age contribute vs visual region?”

4. CAM + SHAP Joint Panels
Create composite images showing:

CAMs on left

SHAP bar chart on right

Prediction label & softmax confidence This tells a full interpretability story in one glance.